# BRCA all omics benchmark
- benchmark for the brca dataset
- take
    - [x] mRNA
    - [x] CNA
    - [x] miRNA
    - [ ] DNA methylation data
- benchmark several prediction methods on them, using
    - [ ] early integration
    - [ ] late integration

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variational_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_lin_eval, svm_rbf_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

In [5]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

In [6]:
labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
labels

sampleID,PAM50_mRNA_nature2012
str,str
"""TCGA-A1-A0SD-0…","""Luminal A"""
"""TCGA-A1-A0SE-0…","""Luminal A"""
"""TCGA-A1-A0SH-0…","""Luminal A"""
"""TCGA-A1-A0SJ-0…","""Luminal A"""
"""TCGA-A1-A0SK-0…","""Basal-like"""
…,…
"""TCGA-E2-A1B4-0…","""Luminal A"""
"""TCGA-E2-A1B5-0…","""Basal-like"""
"""TCGA-E2-A1B6-0…","""Luminal A"""


In [7]:
# ensure that the omic channels are alined with the labels and with each other
assert mrna.columns[1:] == cna.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [26]:
# TODO, join genes with identical features into a single vertex
def find_identical_rows(matrix):
    # Convert the matrix to a structured array
    structured_array = np.core.records.fromarrays(matrix.transpose())
    
    # Find unique rows and their corresponding labels
    _, labels = np.unique(structured_array, return_inverse=True)
    
    # Group indices by labels
    unique_labels, indices = np.unique(labels, return_index=True)
    identical_rows_indices = [np.where(labels == label)[0] for label in unique_labels]
    
    return identical_rows_indices

# Example usage
matrix = np.array([[1, 2], [3, 4], [1, 2], [5, 6], [3, 4]])
identical_rows_indices = find_identical_rows(matrix)

for indices in identical_rows_indices:
    print(f"Identical rows at indices: {indices}")

# identical_rows = find_identical_rows(cna_m)

Identical rows at indices: [0 2]
Identical rows at indices: [1 4]
Identical rows at indices: [3]


In [8]:
# apply individual feature selection

X_mrna = mrna[:,1:].to_numpy().T
X_cna = cna[:,1:].to_numpy().T
X_mirna = mirna[:,1:].to_numpy().T

# mrna_mask, mrna_idx = variational_selection(X_mrna, y, 200)
# cna_mask, cna_idx = variational_selection(X_cna, y, 200)
# mirna_mask, mirna_idx = variational_selection(X_mirna, y, 100)

# mrna_mask.shape
# X_mrna.shape, X_cna.shape
X = np.hstack([X_mrna, X_cna, X_mirna])
X.shape
# X_mrna = X_mrna[:, mrna_mask]
# X_cna = X_cna[:, cna_mask]
# X_mirna = X_mirna[:, mirna_mask]

# # concat the features
# X = np.hstack([X_mrna, X_cna, X_mirna])
# X.shape

(483, 43982)

In [9]:
# run benchmarks
knn_eval(X, y, n_features=1000)

| KNN | 0.66 +/- 0.02 | 0.56 +/- 0.04 | 0.62 +/- 0.03 |
study.best_value=0.6167785914904695, study.best_params={'n_neighbors': 1}


In [63]:
svm_lin_eval(X, y, n_features=1000)

Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
| LIN SVM | 0.82 +/- 0.02 | 0.81 +/- 0.02 | 0.82 +/- 0.02 |
study.best_value=0.82394274728502, study.best_params={'C': 0.0012062927436782822, 'class_weight': None}


{'acc': 0.8234482758620689,
 'f1_macro': 0.8146722196381695,
 'f1_weighted': 0.82394274728502,
 'acc_std': 0.024128076806256414,
 'f1_macro_std': 0.024158775154244774,
 'f1_weighted_std': 0.023747662716833852}

In [64]:
svm_rbf_eval(X, y, n_features=1000)

Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
Pruning trial
| RBF SVM | 0.81 +/- 0.02 | 0.79 +/- 0.02 | 0.80 +/- 0.02 |
study.best_value=0.8024684394738328, study.best_params={'C': 8.681172078950329, 'gamma': 0.0010065527711077555, 'class_weight': 'balanced'}


{'acc': 0.8060344827586208,
 'f1_macro': 0.7858921149649299,
 'f1_weighted': 0.8024684394738328,
 'acc_std': 0.02023568033500948,
 'f1_macro_std': 0.021351915435578694,
 'f1_weighted_std': 0.021399159772430745}

In [65]:
xgboost_eval(X, y, n_features=1000)

0 / 100
1 / 100
Pruning trial
2 / 100
3 / 100
Pruning trial
4 / 100
Pruning trial
5 / 100
Pruning trial
6 / 100
Pruning trial
7 / 100
Pruning trial
8 / 100
Pruning trial
9 / 100
Pruning trial
10 / 100
11 / 100
12 / 100
13 / 100
14 / 100
15 / 100
16 / 100
17 / 100
18 / 100
Pruning trial
19 / 100
Pruning trial
20 / 100
Pruning trial
21 / 100
22 / 100
Pruning trial
23 / 100
24 / 100
Pruning trial
25 / 100
Pruning trial
26 / 100
Pruning trial
27 / 100
Pruning trial
28 / 100
Pruning trial
29 / 100
30 / 100
31 / 100
32 / 100
33 / 100
34 / 100
35 / 100
36 / 100
37 / 100
38 / 100
Pruning trial
39 / 100
Pruning trial
40 / 100
Pruning trial
41 / 100
Pruning trial
42 / 100
43 / 100
44 / 100
45 / 100
Pruning trial
46 / 100
Pruning trial
47 / 100
48 / 100
49 / 100
50 / 100
Pruning trial
51 / 100
52 / 100
53 / 100
Pruning trial
54 / 100
55 / 100
56 / 100
Pruning trial
57 / 100
58 / 100
Pruning trial
59 / 100
60 / 100
Pruning trial
61 / 100
62 / 100
63 / 100
64 / 100
65 / 100
66 / 100
Pruning trial
6

In [73]:
# mlp
train_temp_idx, test_idx = train_test_split(
    np.arange(X.shape[0]),
    test_size=0.2,
    stratify=y,
    random_state=3
)
train_idx, val_idx = train_test_split(
    train_temp_idx, 
    test_size=0.2,
    stratify=y[train_temp_idx],                   
    random_state=3
)

X_train = X[train_idx]
y_train = y[train_idx]
X_train = X[train_idx]
y_train = y[train_idx]
X_train = X[train_idx]
y_train = y[train_idx]

# split data


In [12]:
mlp_eval(
    X, y, n_trials=10, n_features=1000
)

Trial 0 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 1 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Pruning trial after 3 evals, cause 0.24477333182690328 < 0.6929277812403087
Trial 2 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Pruning trial after 3 evals, cause 0.24477333182690328 < 0.6929277812403087
Trial 3 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 4 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Pruning trial after 3 evals, cause 0.24477333182690328 < 0.8060128728645463
Trial 5 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 6 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Pruning trial after 3 evals, cause 0.45156196415833233 < 0.805906020744475
Trial 7 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Pruning trial after 3 evals, cause 0.47118723964673653 < 0.805906020744475
Trial 8 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 9 / 10
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 /

In [26]:
lin_layer = pyg.nn.Linear(10, 5)

In [28]:
lin_layer.weight, lin_layer.bias

(Parameter containing:
 tensor([[-1.9807e-01, -2.3515e-01,  5.0242e-02, -1.2687e-01,  2.9227e-01,
          -1.7938e-01, -1.1826e-02, -3.1594e-01, -6.3897e-05, -2.3742e-01],
         [-2.1721e-01, -1.2257e-01, -2.2888e-01,  2.6437e-01,  2.0000e-01,
           2.9481e-01,  5.7955e-02, -2.9256e-01,  1.7416e-01,  2.0089e-02],
         [-2.6839e-01,  2.1913e-01, -3.0587e-01, -3.1317e-01,  7.9843e-03,
          -1.1149e-01,  1.9922e-01,  6.0216e-02, -1.0590e-01,  1.2973e-01],
         [-4.7408e-02, -3.6408e-04,  4.7792e-02,  2.7520e-02, -2.9809e-01,
           2.6686e-01, -1.0666e-01,  2.7008e-01,  7.6309e-02, -1.7063e-01],
         [-1.7959e-01,  1.9875e-01, -2.6073e-02, -2.1565e-01,  6.1326e-02,
           5.1830e-02, -5.0349e-02, -1.7508e-01, -2.0251e-01,  2.3409e-01]],
        requires_grad=True),
 Parameter containing:
 tensor([ 0.0541,  0.0020, -0.2438,  0.2433,  0.1099], requires_grad=True))

In [31]:
with torch.no_grad():
    print(lin_layer.weight.sum(dim=1))

tensor([-0.9622,  0.1502, -0.4885,  0.0654, -0.3033])


In [17]:
for p in lin_layer.parameters():
    print(p)

Parameter containing:
tensor([[-0.2551,  0.1728,  0.1324,  0.2253,  0.1026, -0.1178, -0.2738, -0.1147,
         -0.2954, -0.1799],
        [-0.0171, -0.1440,  0.0675, -0.1376,  0.0964,  0.0256, -0.0412,  0.2668,
         -0.0864, -0.1193],
        [-0.2921, -0.1134,  0.0668, -0.0426, -0.2441,  0.0956, -0.0082, -0.3141,
         -0.0496,  0.1993],
        [-0.1268, -0.2585, -0.1758, -0.2746,  0.2917, -0.0687,  0.1045,  0.0955,
          0.2000,  0.2809],
        [ 0.1283, -0.1413,  0.2374,  0.0594,  0.0823, -0.2244,  0.0285,  0.2927,
         -0.3086, -0.2827],
        [ 0.2092, -0.2935, -0.1830, -0.2830,  0.0358,  0.0202, -0.2294, -0.0385,
         -0.3131, -0.3099],
        [ 0.2380, -0.2986,  0.1024,  0.0444, -0.2105,  0.0752, -0.2319, -0.2640,
         -0.1241,  0.2102],
        [-0.0597, -0.1554, -0.0678, -0.0610,  0.1387,  0.1822,  0.0123, -0.1223,
          0.2632, -0.0558],
        [ 0.0645,  0.1821,  0.2310,  0.2916,  0.1525,  0.1723, -0.2088,  0.2520,
         -0.0409,  0.1085